# Consumer Credit Risk Analysis — Exploratory Risk Analysis

## Objective

Analyze historical LendingClub loan performance to identify borrower and loan characteristics associated with higher default risk.

This notebook evaluates:

- Overall portfolio default performance
- FICO score and default risk
- Credit grade and default risk
- Debt-to-income ratio (DTI)
- Borrower income
- Interest rate
- Loan amount
- Interactions between major risk factors

The goal is to identify meaningful risk patterns before developing the probability-of-default model.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Load Prepared Modeling Data

Load the cleaned modeling population created in the data preparation notebook.

In [5]:
model_df = pd.read_csv(
    "../data/processed/model_data.csv",
    low_memory=False
)

print(model_df.shape)
model_df.head()

(1632364, 26)


,id,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,open_acc,revol_bal,revol_util,total_acc,application_type,default_flag,fico,int_rate_num
0,120122535,12000.0,36 months,7.97%,A,A5,10+ years,OWN,42000.0,Source Verified,2017-09-01,Fully Paid,debt_consolidation,27.74,0.0,715.0,719.0,0.0,9.0,11457.0,37%,16.0,Individual,0,717.0,7.97
1,119183129,10000.0,36 months,9.44%,B,B1,3 years,MORTGAGE,55000.0,Not Verified,2017-09-01,Fully Paid,debt_consolidation,18.79,0.0,695.0,699.0,0.0,7.0,7188.0,57.5%,10.0,Individual,0,697.0,9.44
2,120086024,8000.0,36 months,16.02%,C,C5,< 1 year,MORTGAGE,120000.0,Not Verified,2017-09-01,Fully Paid,debt_consolidation,20.36,0.0,700.0,704.0,1.0,8.0,19015.0,92.3%,34.0,Joint App,0,702.0,16.02
3,119417303,12800.0,36 months,13.59%,C,C2,5 years,RENT,90000.0,Not Verified,2017-09-01,Fully Paid,debt_consolidation,22.63,0.0,660.0,664.0,2.0,10.0,12660.0,86%,23.0,Individual,0,662.0,13.59
4,119997711,15000.0,36 months,13.59%,C,C2,4 years,MORTGAGE,180000.0,Source Verified,2017-09-01,Fully Paid,medical,38.07,0.0,680.0,684.0,0.0,24.0,107214.0,66.6%,50.0,Individual,0,682.0,13.59


## 2. Portfolio Baseline

Establish the overall portfolio size and observed default rate before analyzing individual risk factors.

In [6]:
portfolio_summary = pd.Series({
    "Total Loans": len(model_df),
    "Total Originations": model_df["loan_amnt"].sum(),
    "Average Loan Amount": model_df["loan_amnt"].mean(),
    "Average FICO": model_df["fico"].mean(),
    "Average DTI": model_df["dti"].mean(),
    "Default Rate": model_df["default_flag"].mean()
})

portfolio_summary

Total Loans            1.632364e+06
Total Originations     2.399099e+10
Average Loan Amount    1.469708e+04
Average FICO           6.993565e+02
Average DTI            1.886970e+01
Default Rate           2.004498e-01
dtype: float64

## 3. Default Risk by FICO Score

FICO score is a core measure of borrower credit quality. Loans are grouped into FICO bands to evaluate how observed default rates change as credit quality declines.

In [7]:
fico_bins = [0, 680, 700, 720, 740, 760, float("inf")]
fico_labels = [
    "660-679",
    "680-699",
    "700-719",
    "720-739",
    "740-759",
    "760+"
]

model_df["fico_band"] = pd.cut(
    model_df["fico"],
    bins=fico_bins,
    labels=fico_labels,
    right=False
)

fico_risk = (
    model_df
    .groupby("fico_band", observed=False)
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

fico_risk["default_rate"] = fico_risk["default_rate"] * 100

fico_risk

,loans,defaults,default_rate
fico_band,,,
660-679,546396,136104,24.909406
680-699,425331,92015,21.633739
700-719,299522,53728,17.937914
720-739,167789,24640,14.685110
740-759,88852,10842,12.202314
760+,104474,9878,9.454984


## 4. Default Risk by Credit Grade

Credit grade provides another view of borrower risk. This analysis compares observed default rates across LendingClub grades A through G.

In [8]:
grade_risk = (
    model_df
    .groupby("grade")
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

grade_risk["default_rate"] = grade_risk["default_rate"] * 100

grade_risk

,loans,defaults,default_rate
grade,,,
A,307652,20127,6.542132
B,471945,65545,13.888271
C,469987,107854,22.948294
D,241784,75302,31.144327
E,101398,39509,38.964279
F,30346,14151,46.632176
G,9252,4719,51.005188


## 5. Default Risk by Debt-to-Income Ratio

Debt-to-income ratio (DTI) measures a borrower's monthly debt obligations relative to income. Loans are grouped into DTI bands to evaluate whether borrowers with heavier debt burdens exhibit higher default rates.

In [9]:
dti_bins = [0, 10, 20, 30, 40, float("inf")]
dti_labels = ["0-10", "10-20", "20-30", "30-40", "40+"]

model_df["dti_band"] = pd.cut(
    model_df["dti"],
    bins=dti_bins,
    labels=dti_labels,
    right=False,
    include_lowest=True
)

dti_risk = (
    model_df
    .groupby("dti_band", observed=False)
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

dti_risk["default_rate"] = dti_risk["default_rate"] * 100

dti_risk

,loans,defaults,default_rate
dti_band,,,
0-10,288976,46633,16.137326
10-20,662770,118959,17.948761
20-30,497871,111480,22.391342
30-40,165325,45091,27.274157
40+,16312,4821,29.554929


## 6. Default Risk by Borrower Income

Borrowers are grouped into income segments to evaluate whether default performance varies with annual income.

In [10]:
income_bins = [0, 40000, 60000, 100000, float("inf")]
income_labels = [
    "Low",
    "Lower-Middle",
    "Upper-Middle",
    "High"
]

model_df["income_group"] = pd.cut(
    model_df["annual_inc"],
    bins=income_bins,
    labels=income_labels,
    right=False
)

income_risk = (
    model_df
    .groupby("income_group", observed=False)
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

income_risk["default_rate"] = income_risk["default_rate"] * 100

income_risk

,loans,defaults,default_rate
income_group,,,
Low,250300,59293,23.688773
Lower-Middle,417268,89869,21.537477
Upper-Middle,607363,119462,19.668962
High,357433,58583,16.389925


## 7. Default Risk by Interest Rate

Interest rates reflect the lender's assessment and pricing of borrower risk. Loans are grouped into interest-rate bands to evaluate how observed default performance varies across pricing tiers.

In [11]:
rate_bins = [0, 10, 15, 20, 25, float("inf")]
rate_labels = ["<10%", "10-15%", "15-20%", "20-25%", "25%+"]

model_df["rate_band"] = pd.cut(
    model_df["int_rate_num"],
    bins=rate_bins,
    labels=rate_labels,
    right=False
)

rate_risk = (
    model_df
    .groupby("rate_band", observed=False)
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

rate_risk["default_rate"] = rate_risk["default_rate"] * 100

rate_risk

,loans,defaults,default_rate
rate_band,,,
<10%,480998,40283,8.374879
10-15%,672400,126735,18.848156
15-20%,342732,104330,30.440694
20-25%,96510,37291,38.639519
25%+,39724,18568,46.742523


## 8. Default Risk by Loan Amount

Loan balances are segmented to evaluate whether default performance varies with the size of the credit exposure.

In [12]:
loan_bins = [0, 5000, 10000, 15000, 20000, float("inf")]
loan_labels = [
    "<$5K",
    "$5K-$10K",
    "$10K-$15K",
    "$15K-$20K",
    "$20K+"
]

model_df["loan_amount_band"] = pd.cut(
    model_df["loan_amnt"],
    bins=loan_bins,
    labels=loan_labels,
    right=False
)

loan_risk = (
    model_df
    .groupby("loan_amount_band", observed=False)
    .agg(
        loans=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

loan_risk["default_rate"] = loan_risk["default_rate"] * 100

loan_risk

,loans,defaults,default_rate
loan_amount_band,,,
<$5K,164947,23633,14.327633
$5K-$10K,378501,61688,16.297975
$10K-$15K,375885,78371,20.849728
$15K-$20K,251733,55783,22.159590
$20K+,461298,107732,23.354101


## 9. Combined Risk Segmentation: FICO × DTI

FICO and DTI capture different dimensions of borrower risk. Combining the two provides a more granular view of how credit quality and debt burden interact.

The table below shows observed default rates for each FICO and DTI combination.

In [13]:
fico_dti_risk = pd.pivot_table(
    model_df,
    values="default_flag",
    index="fico_band",
    columns="dti_band",
    aggfunc="mean",
    observed=False
) * 100

fico_dti_risk.round(1)

dti_band,0-10,10-20,20-30,30-40,40+
fico_band,,,,,
660-679,19.4,22.8,28.1,32.9,36.2
680-699,17.5,19.4,23.7,28.9,31.6
700-719,15.1,15.6,19.5,24.3,28.2
720-739,13.0,12.4,16.1,20.9,25.0
740-759,11.9,10.2,13.3,17.2,21.4
760+,9.8,7.7,10.2,14.6,16.9


## 10. Key Findings

The exploratory analysis identifies several clear patterns in historical loan performance:

- **Credit quality strongly differentiates risk.** Default rates increase substantially as FICO scores decline and LendingClub credit grades deteriorate.
- **Higher debt burdens are associated with higher default risk.** Default rates rise consistently across higher DTI bands.
- **Income provides additional risk differentiation.** Higher-income borrowers experienced lower observed default rates.
- **Interest rate is strongly associated with default performance.** Higher-rate loans experienced materially higher default rates, reflecting the relationship between risk-based pricing and borrower credit quality.
- **Larger loan balances exhibited higher observed default rates.**
- **Risk factors become more informative when combined.** The FICO × DTI analysis shows that borrowers with both weaker credit scores and higher debt burdens experienced the highest default rates.

These findings support using multiple borrower and loan characteristics rather than relying on any single variable when estimating probability of default.